# TAE-IA · Module 6 · L07 — Image Restoration and Super-Resolution

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Lesson** | L07 |
| **Track** | A — Vision |
| **Estimated duration** | 2 hours |
| **GPU required** | T4 (Colab) |
| **Prerequisites** | L06 completed |

## Learning objectives
By the end of this notebook you will be able to:
- [ ] Apply Real-ESRGAN to restore and upscale degraded images
- [ ] Artificially degrade images with noise, JPEG compression, and blur to create ground truth pairs
- [ ] Compute and interpret PSNR and SSIM for quantitative evaluation
- [ ] Identify hallucinated details in restored images and reason about their acceptability

## Before you start
- T4 GPU runtime selected (`Runtime > Change runtime type > T4 GPU`)
- No large model downloads from previous lessons required for this lesson

---

## Cell 0 — Setup (always run this first)

In [ ]:
# ================================================================
# TAE-IA M6 · Standard Setup -- DO NOT MODIFY THIS CELL
# ================================================================
import os, sys, random, time, shutil
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

MODEL_CACHE = '/content/drive/MyDrive/TAE_IA_M6/models'   # persistent, survives sessions
LOCAL_CACHE = '/content/model_cache'                      # ephemeral, dies with the runtime
os.makedirs(MODEL_CACHE, exist_ok=True)
os.makedirs(LOCAL_CACHE, exist_ok=True)

# Clear any cache left by earlier course versions. HF_HOME used to point here,
# and huggingface_hub's symlinked cache is not reliable on Drive's FUSE mount
# (intermittent OSError 95). Nothing in this lab reads those variables now --
# Real-ESRGAN weights are fetched to an explicit folder below.
for _leftover in ('hub', 'xet'):
    _p = os.path.join(MODEL_CACHE, _leftover)
    if os.path.exists(_p):
        shutil.rmtree(_p)

# ----------------------------------------------------------------
# Model cache with automatic fallback
# ----------------------------------------------------------------
# Models are cached on Drive so they survive a session restart. A free
# Google account has only 15 GB, shared with Gmail and Photos, which is
# less than this course needs end to end. So when Drive has no room we
# cache on the runtime's own disk instead: the lab still runs, but that
# copy is deleted when the session ends.
#
# The choice is made per model, not once per notebook. A model already
# sitting on Drive keeps being read from Drive even when Drive is far
# too full to accept anything new.

DRIVE_HEADROOM_GB = 1.0    # never fill Drive to the last byte
SIZE_MARGIN       = 1.15   # temp files and metadata written during a download


def _free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return 0.0


def _is_cached(path):
    """True if <path> holds anything but HuggingFace's incomplete-download folder."""
    return os.path.isdir(path) and any(e != '.cache' for e in os.listdir(path))


def _disk_full(exc):
    """ENOSPC, quota exceeded, or the I/O error drivefs raises when Drive is full."""
    return (getattr(exc, 'errno', None) in (28, 122, 5)
            or 'no space' in str(exc).lower()
            or 'quota' in str(exc).lower())


def cache_dir(name, needed_gb):
    """Return the directory model <name> should live in, preferring Drive."""
    drive_dir = os.path.join(MODEL_CACHE, name)
    local_dir = os.path.join(LOCAL_CACHE, name)

    if _is_cached(drive_dir):
        return drive_dir            # already on Drive: reading it costs no space
    if _is_cached(local_dir):
        return local_dir            # already fell back earlier this session

    required = needed_gb * SIZE_MARGIN + DRIVE_HEADROOM_GB
    free = _free_gb(MODEL_CACHE)
    if free >= required:
        os.makedirs(drive_dir, exist_ok=True)
        print(f'[cache] {name} -> Drive ({needed_gb:.2f} GB needed, {free:.1f} GB free).')
        return drive_dir

    local_free = _free_gb('/content')
    if local_free < required:
        raise RuntimeError(
            f'{name} needs ~{needed_gb:.1f} GB, but only {free:.1f} GB is free on '
            f'Drive and {local_free:.1f} GB on the runtime disk.\n'
            f'Free space in Google Drive (delete unused TAE_IA_M6/models subfolders) '
            f'and re-run this cell.')

    os.makedirs(local_dir, exist_ok=True)
    print(f'[cache] Not enough room on Drive for {name}: needs {needed_gb:.2f} GB\n'
          f'        plus margin, {free:.1f} GB free. Caching on the runtime instead.\n'
          f'        The lab runs normally, but this copy is deleted when the session\n'
          f'        ends and downloads again next time. Free space in Drive to avoid\n'
          f'        the repeat download.')
    return local_dir


def cached_fetch(name, needed_gb, download):
    """Run download(target_dir) in the best available cache and return its result.

    Retries on the runtime disk if Drive fills up mid-download: a pre-flight
    space check cannot catch a quota that runs out halfway through.
    """
    target = cache_dir(name, needed_gb)
    try:
        return download(target)
    except OSError as e:
        if not _disk_full(e) or target.startswith(LOCAL_CACHE):
            raise
        print(f'[cache] Drive ran out of room mid-download ({e}).\n'
              f'        Discarding the partial copy and retrying on the runtime disk.')
        shutil.rmtree(target, ignore_errors=True)
        fallback = os.path.join(LOCAL_CACHE, name)
        os.makedirs(fallback, exist_ok=True)
        return download(fallback)


def cached_snapshot(name, repo_id, needed_gb, **kwargs):
    """snapshot_download into the best available cache, returning its path."""
    from huggingface_hub import snapshot_download

    def _dl(target):
        snapshot_download(repo_id, local_dir=target, **kwargs)
        return target

    return cached_fetch(name, needed_gb, _dl)


_drive_free = _free_gb(MODEL_CACHE)
print(f'Model cache: {MODEL_CACHE}  ({_drive_free:.1f} GB free on Drive)')
if _drive_free < 1.0:
    print('WARNING: under 1 GB free on Drive. Models will cache on the runtime,\n'
          '         but saving your lab outputs may fail. Free space in Drive.')

import torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Go to: Runtime > Change runtime type > T4 GPU')
    raise SystemExit('GPU required.')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print(f'Fixed seed: {SEED}')
print(f'Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}')

# This lab loads no gated HuggingFace model, so no HF login cell is needed.

In [ ]:
# ================================================================
# Install dependencies for L07
# ================================================================
# basicsr's last release (1.4.2, Aug 2022) predates the Python that Colab now
# runs, ships only as a source tarball, and no longer installs as published. We
# download that tarball, apply three fixes to the source, and install from it.
#
# 1. setup.py declares setup_requires=['cython', 'numpy', 'torch'], so setuptools
#    tries to download and build those three before it will emit any package
#    metadata -- pip reports 'metadata-generation-failed'. Nothing needs them at
#    build time: basicsr is pure Python unless you ask for its CUDA ops.
# 2. setup.py's get_version() runs exec() and then reads the result out of
#    locals(). PEP 667 (Python 3.13) made locals() return an independent
#    snapshot, so names defined by exec() are no longer visible there and the
#    build dies with KeyError: '__version__'. This one is fatal on every
#    install path, with or without pip's isolated build environment.
# 3. basicsr/data/degradations.py imports
#        from torchvision.transforms.functional_tensor import rgb_to_grayscale
#    and torchvision deleted that module in 0.17. The function still exists, it
#    just moved to torchvision.transforms.functional.
import hashlib, pathlib, re, subprocess, sys, tarfile, urllib.request

BASICSR_URL = 'https://files.pythonhosted.org/packages/source/b/basicsr/basicsr-1.4.2.tar.gz'
BASICSR_SHA = 'b89b595a87ef964cda9913b4d99380ddb6554c965577c0c10cb7b78e31301e87'

# setup.py's version helper, and the PEP 667 safe rewrite of it.
OLD_GET_VERSION = """def get_version():
    with open(version_file, 'r') as f:
        exec(compile(f.read(), version_file, 'exec'))
    return locals()['__version__']"""

NEW_GET_VERSION = """def get_version():
    ns = {}
    with open(version_file, 'r') as f:
        exec(compile(f.read(), version_file, 'exec'), ns)
    return ns['__version__']"""


def pip_install(*args):
    """Run pip and show its output -- Colab sends subprocess output to the
    runtime log, not to the cell, so a failure would otherwise be silent."""
    done = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                          capture_output=True, text=True)
    if done.returncode:
        print(done.stdout, done.stderr, sep='\n')
        raise RuntimeError(f"pip install {' '.join(args)} failed (exit {done.returncode})")


def install_basicsr(build_dir='/content/basicsr_build'):
    build = pathlib.Path(build_dir)
    build.mkdir(parents=True, exist_ok=True)
    tgz = build / 'basicsr-1.4.2.tar.gz'
    if not tgz.exists():
        urllib.request.urlretrieve(BASICSR_URL, tgz)
    digest = hashlib.sha256(tgz.read_bytes()).hexdigest()
    if digest != BASICSR_SHA:
        raise RuntimeError(f'basicsr tarball checksum mismatch: {digest}')
    with tarfile.open(tgz) as tar:
        try:
            tar.extractall(build, filter='data')   # filter= requires Python 3.12+
        except TypeError:
            tar.extractall(build)
    src = build / 'basicsr-1.4.2'

    setup_py = src / 'setup.py'
    text = setup_py.read_text(encoding='utf-8')
    text = re.sub(r'^\s*setup_requires=.*\n', '', text, flags=re.M)   # fix 1
    text = text.replace(OLD_GET_VERSION, NEW_GET_VERSION)             # fix 2
    if 'setup_requires' in text or "locals()['__version__']" in text:
        raise RuntimeError('basicsr setup.py is not the expected 1.4.2 source')
    setup_py.write_text(text, encoding='utf-8')

    deg = src / 'basicsr' / 'data' / 'degradations.py'                 # fix 3
    deg.write_text(
        deg.read_text(encoding='utf-8').replace('torchvision.transforms.functional_tensor',
                                                'torchvision.transforms.functional'),
        encoding='utf-8')

    # --no-deps: basicsr's requirements.txt pins tb-nightly, which would displace
    # the tensorboard Colab ships. We install the pieces it actually imports.
    pip_install('addict', 'lmdb', 'yapf')
    pip_install('--no-deps', str(src))


install_basicsr()

# realesrgan itself is a wheel. --no-deps stops pip from pulling gfpgan and
# facexlib, which this lab never uses: RealESRGANer needs only basicsr + torch.
pip_install('--no-deps', 'realesrgan')

import basicsr, realesrgan
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
print(f'basicsr {basicsr.__version__} + realesrgan {realesrgan.version.__version__} ready.')


---
## Part 1 — Context and Key Concepts

> Read this before running any code.

### What is blind super-resolution?

Classical super-resolution models assume a known, clean degradation (typically bicubic downsampling). Real photos are degraded by combinations of noise, JPEG compression, blur, and lossy resize — often applied multiple times. A model trained only on bicubic downsampling fails on real images.

Real-ESRGAN addresses this with a **high-order degradation pipeline**: during training it randomly applies sequences of blur → resize → noise → JPEG compression, creating highly varied synthetic degradations. The resulting model can handle most real-world degradation without knowing its exact type or severity.

### PSNR and SSIM: interpreting the numbers

Both metrics require a reference (ground truth) image.

- **PSNR** (dB): 30–35 = acceptable, 35–40 = good, >40 = excellent. Not always correlated with perceived sharpness.
- **SSIM**: 0.85–0.95 = good structural similarity. More perceptually relevant than PSNR.

**Critical limitation:** both metrics can decrease when the model hallucinations plausible-looking but incorrect detail. A sharper, more visually appealing image can score *lower* on PSNR than a blurry one that is closer to the ground truth pixel-by-pixel.

### Hallucination in super-resolution

At 4× upscaling, 15 out of every 16 pixels in the output are generated by the model — not measured from the input. The model fills these with the most statistically plausible texture given the context. This works well for natural textures (grass, brick, fur) but fails on text, faces, and structured patterns where the "most plausible" guess may be factually wrong.

---

## Part 2 — Lab

### Section 2.0 — Load Real-ESRGAN

In [ ]:
# Section 2.0 — Load the Real-ESRGAN 4× upsampler
import os, time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import requests

from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.utils.download_util import load_file_from_url

OUTPUT_DIR = '/content/drive/MyDrive/TAE_IA_M6/L07_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Weights come from the author's GitHub releases, not HuggingFace, so there is
# no snapshot_download here. We fetch them into Drive explicitly: left to
# itself RealESRGANer downloads into its own site-packages folder, which is
# wiped on every runtime reset.
WEIGHTS_DIR = cache_dir('realesrgan', 0.13)

WEIGHT_URLS = {
    4: 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    2: 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth',
}


def load_upsampler(scale=4, tile=256):
    """Build a Real-ESRGAN upsampler at the given scale.

    tile: process the image in tile x tile blocks so a large input does not
    exhaust T4 VRAM. tile=0 disables tiling (Exercise 2 explores the cost).
    """
    weights_path = cached_fetch(
        'realesrgan', 0.13,
        lambda d: load_file_from_url(
            url=WEIGHT_URLS[scale], model_dir=d, progress=True, file_name=None))
    model = RRDBNet(
        num_in_ch=3, num_out_ch=3,
        num_feat=64, num_block=23,
        num_grow_ch=32, scale=scale
    )
    return RealESRGANer(
        scale=scale,
        model_path=weights_path,
        model=model,
        tile=tile,
        tile_pad=10,
        pre_pad=0,
        half=True
    )


upsampler_4x = load_upsampler(scale=4)
print(f'Real-ESRGAN 4× loaded. Weights cached in {WEIGHTS_DIR}')

### Section 2.1 — Download the ground-truth source image

In [ ]:
# Section 2.1 — Download the ground-truth source image
#
# Wikimedia rejects the default requests User-Agent with HTTP 403, and only
# serves a fixed set of thumbnail widths (20/40/60/120/250/330/500/960/1280/
# 1920/3840) -- any other width returns HTTP 400. Both were silently swallowed
# by the old try/except here, which substituted a synthetic noise image and
# made every PSNR number below meaningless.
WIKI_UA = {'User-Agent': 'TAE-IA-M6-course/1.0 (classroom use; contact instructor)'}

# A macro photograph: fine, irregular texture is what makes super-resolution
# results interesting to look at. Students can swap in any detailed image.
SOURCE_URL = ('https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/'
              'Camponotus_flavomarginatus_ant.jpg/500px-Camponotus_flavomarginatus_ant.jpg')

resp = requests.get(SOURCE_URL, timeout=20, headers=WIKI_UA)
resp.raise_for_status()
source = Image.open(BytesIO(resp.content)).convert('RGB').resize((256, 256), Image.LANCZOS)
print(f'Source image loaded: {source.size}')

source.save(os.path.join(OUTPUT_DIR, 'source_gt.png'))
plt.figure(figsize=(3, 3))
plt.imshow(source); plt.title(f'Ground truth  {source.size}'); plt.axis('off')
plt.tight_layout(); plt.show()

### Section 2.2 — Degrade and restore with ground truth

In [ ]:
# Section 2.2 — Create controlled degradations
import cv2

src_np = np.array(source)   # uint8, (256, 256, 3)

# Degradation 1: Gaussian noise (σ=25)
rng   = np.random.default_rng(SEED)
noise = rng.normal(0, 25, src_np.shape).astype(np.int16)
deg_noise = Image.fromarray(np.clip(src_np.astype(np.int16) + noise, 0, 255).astype(np.uint8))

# Degradation 2: JPEG compression (quality=8)
buf      = BytesIO()
source.save(buf, format="JPEG", quality=8)
deg_jpeg = Image.open(BytesIO(buf.getvalue())).convert("RGB")

# Degradation 3: Gaussian blur (kernel 9×9, σ=3)
deg_blur = Image.fromarray(cv2.GaussianBlur(src_np, (9, 9), 3))

# Degradation 4: Low resolution (4× downsample → 64×64, then bicubic back to 256×256)
lr_size  = (source.width // 4, source.height // 4)
deg_lr   = source.resize(lr_size, Image.BICUBIC).resize((256, 256), Image.BICUBIC)

degradations = [
    ('noise',  deg_noise),
    ('jpeg',   deg_jpeg),
    ('blur',   deg_blur),
    ('lowres', deg_lr),
]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (title, img) in zip(axes, [('Ground truth', source)] + degradations):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.suptitle('Source and four controlled degradations', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Section 2.2 (cont.) — Restore each degradation and measure PSNR/SSIM
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity  as compute_ssim

gt_np = np.array(source)

def evaluate(original_np, restored_np):
    p = compute_psnr(original_np, restored_np, data_range=255)
    s = compute_ssim(original_np, restored_np, channel_axis=2, data_range=255)
    return p, s

results = {}
print(f"{'Degradation':<12} | {'Time':>5} | {'PSNR (dB)':>10} | {'SSIM':>6}")
print("-" * 42)

for name, deg_img in degradations:
    deg_np = np.array(deg_img)

    # Baseline: degraded image vs. ground truth (before restoration)
    psnr_deg, ssim_deg = evaluate(gt_np, deg_np)

    # Restore with Real-ESRGAN 4×, then bicubic back to 256×256 for fair comparison
    t0           = time.time()
    out_np, _    = upsampler_4x.enhance(deg_np, outscale=4)
    t            = time.time() - t0
    restored_img = Image.fromarray(out_np).resize((256, 256), Image.LANCZOS)
    restored_np  = np.array(restored_img)

    psnr_rest, ssim_rest = evaluate(gt_np, restored_np)

    results[name] = dict(
        deg_img=deg_img, restored_img=restored_img,
        psnr_deg=psnr_deg, ssim_deg=ssim_deg,
        psnr_rest=psnr_rest, ssim_rest=ssim_rest, time=t
    )
    print(f"{name:<12} | {t:>4.1f}s | {psnr_rest:>9.2f} dB | {ssim_rest:>5.3f}  "
          f"  (degraded: {psnr_deg:.1f} dB / {ssim_deg:.3f})")

# Display 3-column grid: degraded | restored | ground truth
fig, axes = plt.subplots(len(degradations), 3, figsize=(12, 4 * len(degradations)))
for row, (name, r) in enumerate(results.items()):
    axes[row, 0].imshow(r['deg_img']);
    axes[row, 0].set_title(f"Degraded ({name})\nPSNR {r['psnr_deg']:.1f} dB  SSIM {r['ssim_deg']:.3f}", fontsize=9)
    axes[row, 0].axis('off')
    axes[row, 1].imshow(r['restored_img'])
    axes[row, 1].set_title(f"Restored\nPSNR {r['psnr_rest']:.1f} dB  SSIM {r['ssim_rest']:.3f}", fontsize=9)
    axes[row, 1].axis('off')
    axes[row, 2].imshow(source);   axes[row, 2].set_title('Ground truth'); axes[row, 2].axis('off')
plt.suptitle('Degraded → Restored → Ground truth', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'restoration_comparison.png'), dpi=100)
plt.show()

**What do you observe?**  
- Which degradation type showed the largest PSNR improvement after restoration?
- For which degradation type was the perceptual quality improvement most visible despite the numbers?
- Is there a case where the restored image looks worse than the degraded one?

*Write your observation here:*

(double-click to edit)

### Section 2.3 — Scale factor comparison: 2× vs. 4×

In [ ]:
# Section 2.3 — Load 2× model and compare
upsampler_2x = load_upsampler(scale=2)
print("Real-ESRGAN 2× loaded.")

# Use the JPEG-degraded version as input (good stress test)
deg_np = np.array(deg_jpeg)

# 2× output
t0          = time.time()
out_2x, _   = upsampler_2x.enhance(deg_np, outscale=2)
t_2x        = time.time() - t0
img_2x      = Image.fromarray(out_2x)                             # 512×512
img_2x_down = img_2x.resize((256, 256), Image.LANCZOS)            # back to 256 for metric
psnr_2x, ssim_2x = evaluate(gt_np, np.array(img_2x_down))

# 4× output
t0          = time.time()
out_4x, _   = upsampler_4x.enhance(deg_np, outscale=4)
t_4x        = time.time() - t0
img_4x      = Image.fromarray(out_4x)                             # 1024×1024
img_4x_down = img_4x.resize((256, 256), Image.LANCZOS)            # back to 256 for metric
psnr_4x, ssim_4x = evaluate(gt_np, np.array(img_4x_down))

print(f"2× | output: {img_2x.size} | {t_2x:.1f}s | PSNR {psnr_2x:.2f} dB | SSIM {ssim_2x:.3f}")
print(f"4× | output: {img_4x.size} | {t_4x:.1f}s | PSNR {psnr_4x:.2f} dB | SSIM {ssim_4x:.3f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (title, img) in zip(axes, [
    ('Degraded (JPEG q=8)',      deg_jpeg),
    (f'2× ({img_2x.size[0]}px)\n{t_2x:.1f}s  PSNR {psnr_2x:.1f}', img_2x_down),
    (f'4× ({img_4x.size[0]}px)\n{t_4x:.1f}s  PSNR {psnr_4x:.1f}', img_4x_down),
    ('Ground truth',             source),
]):
    ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis('off')
plt.suptitle('Scale factor comparison: 2× vs. 4× (JPEG degradation)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'scale_comparison.png'), dpi=100)
plt.show()

**What do you observe?**  
- Does 4× produce a higher PSNR than 2× when both are downsampled to the same 256×256 for comparison?
- Visually, which upscaling factor looks sharper at the native output resolution?
- How does the 4× time compare to 2× — is the ratio proportional to pixel count (4×) or closer to 2×?

*Write your observation here:*

(double-click to edit)

### Section 2.4 — Restore a real degraded photo (no ground truth)

In [ ]:
# Section 2.4 — Real photo restoration (no ground truth available)
# Students: replace PHOTO_URL with any low-quality or old photo, or upload a
# file and pass its path to Image.open() instead.
#
# 320px is not one of Wikimedia's supported thumbnail widths, so the old URL
# here returned HTTP 400 and the except branch quietly substituted a
# re-degraded copy of the Section 2.1 source -- which meant Part 4.2 asked you
# to hunt for hallucinations in a photo that was never a real photo.
PHOTO_URL = ('https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/'
             'Bikesgray.jpg/250px-Bikesgray.jpg')

resp = requests.get(PHOTO_URL, timeout=20, headers=WIKI_UA)
resp.raise_for_status()
photo = Image.open(BytesIO(resp.content)).convert('RGB')
print(f'Photo downloaded: {photo.size}')

photo.save(os.path.join(OUTPUT_DIR, 'real_photo_original.png'))

photo_np      = np.array(photo)
t0            = time.time()
out_np, _     = upsampler_4x.enhance(photo_np, outscale=4)
t_real        = time.time() - t0
restored_real = Image.fromarray(out_np)

print(f'Restored: {photo.size} → {restored_real.size}  |  {t_real:.1f}s')
restored_real.save(os.path.join(OUTPUT_DIR, 'real_photo_restored.png'))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(photo);         axes[0].set_title(f'Original  {photo.size}'); axes[0].axis('off')
axes[1].imshow(restored_real); axes[1].set_title(f'Restored 4×  {restored_real.size}\n{t_real:.1f}s'); axes[1].axis('off')
plt.suptitle('Real photo restoration — no ground truth', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'real_photo_comparison.png'), dpi=100)
plt.show()

In [ ]:
# Section 2.4 (cont.) — Zoom into a detail region to inspect hallucination
# Crop a 64×64 patch from both original and restored (scaled to same display size)

# Adjust these coordinates to a region of interest in your image
CROP_X, CROP_Y = 50, 50   # top-left corner in original pixels
CROP_W, CROP_H = 64, 64

patch_orig = photo.crop((CROP_X, CROP_Y, CROP_X + CROP_W, CROP_Y + CROP_H))
# Corresponding region in 4× restored image
patch_rest = restored_real.crop((
    CROP_X * 4, CROP_Y * 4,
    (CROP_X + CROP_W) * 4, (CROP_Y + CROP_H) * 4
))

# Bicubic upscale of original patch for comparison
patch_bicubic = patch_orig.resize(patch_rest.size, Image.BICUBIC)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, img) in zip(axes, [
    ('Original patch (native res)', patch_orig),
    ('Bicubic 4× (no model)',        patch_bicubic),
    ('Real-ESRGAN 4×',               patch_rest),
]):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')
plt.suptitle(f'Detail zoom — patch ({CROP_X},{CROP_Y}) size {CROP_W}×{CROP_H}', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hallucination_zoom.png'), dpi=120)
plt.show()

print("Tip: Look for textures that appear too regular, faces with altered features, or text that changed.")

**What do you observe?**  
- Identify one specific region where Real-ESRGAN added plausible detail (looks better than bicubic).
- Identify one specific region where the model appears to have invented detail that may not be accurate.
- How could you verify whether the invented detail is correct?

*Write your observation here (reference specific regions in the zoomed image):*

(double-click to edit)

---
## Part 3 — Exercises

### Exercise 1 — Combined degradation

**Task:** Create a "combined" degradation: apply Gaussian blur (5×5), then JPEG compression at quality=20, then add Gaussian noise (σ=15). Restore with Real-ESRGAN 4× and compute PSNR/SSIM vs. ground truth. Compare the result to the individual degradations from Section 2.2.

**Expected output:** side-by-side of combined degraded, restored, and ground truth with metric labels.

In [ ]:
# Exercise 1 -- Combined degradation and restoration
# Remember: save any images to OUTPUT_DIR, e.g. img.save(os.path.join(OUTPUT_DIR, "exercise1.png"))
# ...

*How does the PSNR for combined degradation compare to the individual degradations?  
Is this what you expected?*

(double-click to edit)

### Exercise 2 — Tile size effect on speed and quality

**Task:** Re-run Real-ESRGAN 4× on the same JPEG-degraded image with `tile=0` (no tiling, full image at once), `tile=128`, and `tile=256`. Record inference time and PSNR for each. Show any visible tile boundary artifacts when using small tile sizes.

**Expected output:** timing table and side-by-side grid. Note: `tile=0` may OOM on T4 for large images — if it does, document the error and explain why.

In [ ]:
# Exercise 2 -- Tile size sweep
# Remember: save any images to OUTPUT_DIR, e.g. img.save(os.path.join(OUTPUT_DIR, "exercise2.png"))
# ...

*Were tile boundary artifacts visible? At which tile size?  
Was tile=0 possible on T4 for your image size?*

(double-click to edit)

---
## Part 4 — Critical Analysis

> Required. Answer with real outputs from today's session.

**4.1 — PSNR vs. perception:** rank the four degradation types (noise, JPEG, blur, lowres) by PSNR *after* restoration, then rank them by your own visual preference. Are the rankings the same? If not, explain what PSNR failed to capture in the cases where they diverge.

*Write here (include actual PSNR values from Section 2.2):*


---

**4.2 — Hallucination evidence:** in Section 2.4 (real photo), identify one specific region (describe it by location and content, e.g., "upper-left corner, chain links") where the model added plausible detail, and one where it appears to have invented or distorted content. How can you tell the difference between a good reconstruction and a hallucination without ground truth?

*Write here (reference your hallucination_zoom.png):*


---

**4.3 — Scale factor decision:** based on your Section 2.3 results, at what output resolution would you stop using 4× and prefer 2× followed by bicubic upsampling? What specific quality issue appears first as you push to higher scale factors?

*Write here (reference Section 2.3 numbers):*


---

**4.4 — Application boundary:** name one domain where Real-ESRGAN output should **never** be used without explicit disclosure that the image has been modified by AI, and one domain where hallucination is acceptable. Justify both with a concrete scenario, not just a domain name.

*Write here (be specific — e.g., "forensic evidence photos in court" not just "legal"):*


---
## Submission Checklist

- [ ] All cells ran from start to finish without errors
- [ ] Section 2.2: degraded → restored grid with PSNR/SSIM for all 4 degradation types
- [ ] Section 2.3: scale factor comparison (2× vs. 4×) with timing and metrics
- [ ] Section 2.4: real photo before/after comparison saved
- [ ] Section 2.4: hallucination zoom saved
- [ ] Exercise 1: combined degradation with metric comparison
- [ ] Exercise 2: tile size sweep with timing table
- [ ] Part 4: Critical Analysis completed (all 4 questions with specific evidence)
- [ ] Cleanup cell run (see below)

**Save:** `File > Save a copy in Drive`

---
## Drive Cleanup — Run after submitting

Real-ESRGAN weights are not needed in future lessons. Run this cell to free ~130 MB from Drive before L08.

In [ ]:
# Drive cleanup — run after saving and submitting the notebook
import shutil, os, subprocess

to_delete = [
    os.path.join(MODEL_CACHE, 'realesrgan'),   # ~130 MB (2× and 4× weights)
]

for path in to_delete:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f'Deleted: {path}')
    else:
        print(f'Not found (may already be deleted): {path}')

result = subprocess.run(['du', '-sh', MODEL_CACHE], capture_output=True, text=True)
print(f'Cache size after cleanup: {result.stdout.strip()}')

---
## Before You Close This Tab

Once your outputs are saved to Drive, **disconnect and delete the runtime**
(`Runtime > Disconnect and delete runtime`). A Colab session that is merely
*connected* burns GPU quota on the free tier — and compute-unit balance on
Pro — at the same rate whether or not anything is running. Leaving the tab
open after you finish is the single most common way students lose the quota
they need for the next lesson.

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L07*  
*Platform: Google Colab (T4 GPU) · Python 3.10*